# XTR Indexing and Retrieval
In this notebook, we'll guide you through indexing and retrieving documents using a model trained with the approach outlined in [XTR](https://arxiv.org/pdf/2304.01982.pdf).
In orded to run (almost) instantaneously, we use trivial data sizes of training data and collection to search.


## Initial setup

We start by defining variables specifying locations of data we will use:

In [1]:
import os
import tempfile

model_name_or_path = "PrimeQA/XTR-t5-base-40k"
test_files_location = '../../../tests/resources/ir_dense'
with tempfile.TemporaryDirectory() as working_dir:
    output_dir=os.path.join(working_dir, 'output_dir')
    
index_name = 'index_name'

## Indexing

To run indexing, we need an existing model (checkpoint). For this tutorial, we will use the [XTR-t5-base-40k](https://huggingface.co/PrimeQA/XTR-t5-base-40k) model from huggingface.
Next, we will index a collection of documents, using model representaion from the previous step. The collection is a TSV file, containing each document's ID, title, and text.

In [2]:
collection_fn = os.path.join(test_files_location, "xorqa.train_ir_001pct_at_0_pct_collection_fornum.tsv")

Here is an example document:

In [3]:
import pandas as pd
from IPython.display import display, HTML
data = pd.read_csv(collection_fn, sep='\t', header=0, nrows=1)
display(HTML(data.to_html()))

,id,text,title
0,1,"The Kangxi Emperor's reign of 61 years makes him the longest-reigning emperor in Chinese history (although his grandson, the Qianlong Emperor, had the longest period of ""de facto"" power) and one of the longest-reigning rulers in the world. However, since he ascended the throne at the age of seven, actual power was held for six years by four regents and his grandmother, the Grand Empress Dowager Xiaozhuang.",Kangxi Emperor


Next we instantiate the indexer and index the collection:

In [7]:
from primeqa.components.indexer.dense import XTRIndexer

indexer = XTRIndexer(model_name_or_path=model_name_or_path, index_root=None, index_name=index_name)
indexer.load()
indexer.index(collection = collection_fn, overwrite=True)

#> Will delete 11 files already at index_name in 20 seconds...
#> Starting...
No CUDA runtime is found, using CUDA_HOME='/opt/share/cuda-12.1'
{"time":"2024-04-18 08:43:28,149", "name": "faiss.loader", "level": "INFO", "message": "Loading faiss."}
{"time":"2024-04-18 08:43:28,163", "name": "faiss.loader", "level": "INFO", "message": "Successfully loaded faiss."}
#> Loading collection...
0M RANK:0
[Apr 18, 08:43:29] [0] 		 # of sampled PIDs = 6 	 sampled_pids[:3] = [3, 0, 2]
#> Encoding 6 passages..
#> checkpoint, docFromText, Input: Kangxi Emperor | The Kangxi Emperor's reign of 61 years makes him the longest-reigning emperor in Chinese history (although his grandson, the Qianlong Emperor, had the longest period of "de facto" power) and one of the longest-reigning rulers in the world. However, since he ascended the throne at the age of seven, actual power was held for six years by four regents and his grandmother, the Grand Empress Dowager Xiaozhuang., 		 64
#> checkpoint, docFromText,

WARNING clustering 912 points to 64 centroids: please provide at least 2496 training points
0it [00:00, ?it/s]

#> Encoding 6 passages..
[Apr 18, 08:43:33] [0] 		 #> Saving chunk 0: 	 6 passages and 960 embeddings. From #0 onward.
RANK:0
RANK:0
offset: 0
chunk codes size(0): 960
codes size(0): 960
codes size(): torch.Size([960])
RANK:0
RANK:0
>>>>Empty cluster ids: {59, 35}
>>>>partition.size(0): 62
>>>>num_partition: 64
RANK:0
#> Optimizing IVF to store map from centroids to list of pids..
#> Building the emb2pid mapping..
len(emb2pid) = 960
#> Saved IVF to index_name/ivf.eid.pt
#> Saved EMB2PID to index_name/emb2pid.pt
[Apr 18, 08:43:33] [0] 		 #> Saving the indexing metadata to index_name/metadata.json ..
[Apr 18, 08:43:33] [0] 		 #> Saving the empty cluster IDs to index_name/empty_clusters.json ..
RANK:0



1it [00:01,  1.64s/it]
62it [00:00, 308112.38it/s]


#> Joined...


### Search
Next, we use the trained model and the index to search the collection, using queries in the form of a list of strings: (Note: Search is only supported on GPU.)

In [7]:
from primeqa.components.retriever.dense import XTRRetriever

retriever = XTRRetriever(model_name_or_path=model_name_or_path, index_root=None, index_name=index_name, ndocs=2, ncells=2, max_num_documents = 2)
retriever.load()
results = retriever.predict(input_texts = ['Who is Michael Wigge'])

#> Loading codec...
#> Loading IVF...
#> Loading mappings...


Retrieving:: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 73.88it/s]


Here is the top search result for our query, containing document_id and score:

In [12]:
list(results[0])[0]


('6', 217.484375)

Here is the top retrieved document:

In [14]:
with open(collection_fn, 'r') as f:
    for line in f.readlines():
        if str(list(results[0])[0][0]) == line.split()[0]:
            print(line)

6	Michael Wigge Michael Wigge is a travel writer and entertainment personality in Europe and in the United States. His work is characterized by a mixture of journalism and entertainment. His specialties are cultural issues which he examines in a very entertaining way. In 2002, Wigge drew attention to himself in Germany for the first time on TV broadcaster VIVA plus presenting comedy clips on the daily show “London Calling”. In this context he sets a record for the longest donkey ride in music television history and visits the Queen of England, dressed as King Henry VIII, on her 50th throne	Michael Wigge

